# 第03章 B1（G8）验证实验

日期：2026-11-10；轮次：初次验证。受审课程main完整SHA：`c8a8e947b0d9606caf38302279b48c4f9116567d`。
本Notebook由Codex辅助生成并实际执行，尚待学生人工核验签认。未使用B2结论。
参考值来自本文写出的解析解、离散特征模态和独立高阶积分，不以A输出作为真值。
选定A代码原文嵌入下方以保证离线可运行；完整A原Notebook的顺序执行另见本地复现证据。


In [1]:
import ast, json, math, sys, platform, hashlib
from pathlib import Path
import numpy as np
import scipy
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
from scipy.integrate import quad, solve_ivp
from scipy.optimize import minimize_scalar
plt.rcParams['font.sans-serif']=['Microsoft YaHei','DejaVu Sans']
RESULTS={}
FIG=Path('图片');FIG.mkdir(exist_ok=True)
print('Python',sys.version.split()[0], 'numpy',np.__version__,'scipy',scipy.__version__)
def defs(src,ns):
    tree=ast.parse(src)
    tree.body=[n for n in tree.body if isinstance(n,(ast.FunctionDef,ast.ClassDef))]
    exec(compile(tree,'<受审A原始定义>','exec'),ns)
def record(name,rows):
    RESULTS[name]=rows
    print(name);print(json.dumps(rows,ensure_ascii=False,indent=2))

Python 3.12.14 numpy 1.26.4 scipy 1.14.1


## 3.1 Euler与RK4：解析终点、收敛阶及相同函数求值预算
参考：对 $y\prime=-y$，$y(1)=e^{-1}$。观测阶取 $\log_2(E(h)/E(h/2))$，在本测试渐近区间Euler容许0.9—1.2，RK4容许3.8—4.2。函数求值次数通过包装f实际计数。

In [2]:
A1_ODE='import time\n\nfrom IPython.display import HTML\n\nimport numpy as np\nimport pandas as pd\nimport matplotlib.pyplot as plt\n\nplt.rcParams["figure.figsize"] = (5.6, 3.6)\nplt.rcParams["axes.grid"] = True\n\n\ndef f(t, y):\n    return -y\n\n\ndef exact(t):\n    return np.exp(-t)\n\n\ndef euler(t0, y0, T, h):\n    """在 [t0, T] 上以步长 h 做 Euler 迭代，返回终点值与右端函数求值次数。"""\n    M = int(round((T - t0) / h))\n    h = (T - t0) / M                      # 区间等分数为 M，节点数为 M+1\n    y = y0\n    for _ in range(M):\n        y = y + h * f(t0, y)\n    return y, M\n\n\ndef rk4(t0, y0, T, h):\n    """在 [t0, T] 上以步长 h 做经典 RK4 迭代，返回终点值与右端函数求值次数。"""\n    M = int(round((T - t0) / h))\n    h = (T - t0) / M\n    y = y0\n    for _ in range(M):\n        k1 = f(t0, y)\n        k2 = f(t0 + h / 2, y + h * k1 / 2)\n        k3 = f(t0 + h / 2, y + h * k2 / 2)\n        k4 = f(t0 + h, y + h * k3)\n        y = y + h * (k1 + 2 * k2 + 2 * k3 + k4) / 6\n    return y, 4 * M\n\n\ndef trajectory(solver, y0, T, h):\n    """返回 [0, T] 上的数值解轨迹，用于观察符号交替与幅值变化。"""\n    M = int(round(T / h))\n    h = T / M\n    y = np.empty(M + 1)\n    y[0] = y0\n    for i in range(M):\n        y[i + 1] = solver(0.0, y[i], h, h)[0]   # 单步：仅推进一个步长 h\n    return np.linspace(0.0, T, M + 1), y\n\n\ndef cpu_time(solver, t0, y0, T, h, reps=21):\n    """reps 次求解耗时的中位数，单位 ms；取中位数以减小系统调度造成的波动。"""\n    ts = np.empty(reps)\n    for k in range(reps):\n        tic = time.perf_counter()\n        solver(t0, y0, T, h)\n        ts[k] = time.perf_counter() - tic\n    return float(np.median(ts)) * 1e3\n'
A2_ODE="import math\nimport sys\nimport platform\nfrom pathlib import Path\nimport matplotlib\nimport matplotlib.pyplot as plt\n\nOUTPUT_DIR = Path('images')\nOUTPUT_DIR.mkdir(exist_ok=True)\n\ndef f(t, y):\n    return -y\n\ndef exact(t):\n    return math.exp(-t)\n\ndef forward_euler_step(t, y, h):\n    return y + h*f(t, y)\n\ndef backward_euler_step(t, y, h):\n    # Only for f(t,y) = -y. Solve (1+h)*y_next = y.\n    return y/(1+h)\n\ndef rk4_step(t, y, h):\n    k1 = f(t, y)\n    k2 = f(t+h/2, y+h*k1/2)\n    k3 = f(t+h/2, y+h*k2/2)\n    k4 = f(t+h, y+h*k3)\n    return y + h*(k1+2*k2+2*k3+k4)/6\n\nMETHODS = {\n    'Forward Euler': forward_euler_step,\n    'Backward Euler': backward_euler_step,\n    'Classical RK4': rk4_step,\n}\n\ndef solve(method, N, T=2.0, y0=1.0):\n    if not isinstance(N, int) or isinstance(N, bool) or N < 1:\n        raise ValueError('N must be a positive integer')\n    if T <= 0:\n        raise ValueError('T must be positive')\n    h = T/N\n    ts = [n*h for n in range(N+1)]\n    ys = [float(y0)]\n    for n in range(N):\n        ys.append(method(ts[n], ys[-1], h))\n    return ts, ys\n\nprint('Python:', sys.version.split()[0])\nprint('Matplotlib:', matplotlib.__version__)\nprint('Platform:', platform.system(), platform.machine())"

In [3]:
a1={'np':np};defs(A1_ODE,a1)
a2={'math':math};defs(A2_ODE,a2)
ode=[]
for group,ns in [('A1',a1),('A2',a2)]:
    for method in ['Euler','RK4']:
        prev=None
        for N in [16,32,64,128]:
            count=[0]
            def counted(t,y): count[0]+=1;return -y
            ns['f']=counted
            if group=='A1': val,reported=ns['euler' if method=='Euler' else 'rk4'](0.,1.,1.,1/N)
            else:
                step=ns['forward_euler_step' if method=='Euler' else 'rk4_step']
                val=ns['solve'](step,N,T=1.)[1][-1];reported=None
            error=abs(val-math.exp(-1));p=None if prev is None else math.log2(prev/error)
            ode.append(dict(group=group,method=method,N=N,error=error,order=p,nfev=count[0],reported_nfev=reported))
            assert count[0]==N*(1 if method=='Euler' else 4)
            if p is not None: assert (0.9<p<1.2) if method=='Euler' else (3.8<p<4.2)
            prev=error
record('ODE',ode)
budget=[]
for budget_n in [64,128,256]:
    ye=a1['euler'](0.,1.,1.,1/budget_n)[0]
    yr=a1['rk4'](0.,1.,1.,4/budget_n)[0]
    budget.append(dict(nfev=budget_n,euler_error=abs(ye-math.exp(-1)),rk4_error=abs(yr-math.exp(-1))))
record('equal_cost',budget)
for group in ['A1','A2']:
 for method in ['Euler','RK4']:
    rr=[r for r in ode if r['group']==group and r['method']==method]
    plt.loglog([1/r['N'] for r in rr],[r['error'] for r in rr],'o-',label=group+' '+method)
plt.legend();plt.xlabel('h');plt.ylabel('Endpoint absolute error');plt.tight_layout()
plt.savefig(FIG/'B1组-ODE收敛.png');plt.close()

ODE
[
  {
    "group": "A1",
    "method": "Euler",
    "N": 16,
    "error": 0.01180531071964952,
    "order": null,
    "nfev": 16,
    "reported_nfev": 16
  },
  {
    "group": "A1",
    "method": "Euler",
    "N": 32,
    "error": 0.005824151915125697,
    "order": 1.0193161223839076,
    "nfev": 32,
    "reported_nfev": 32
  },
  {
    "group": "A1",
    "method": "Euler",
    "N": 64,
    "error": 0.002892916927534961,
    "order": 1.0095230910767712,
    "nfev": 64,
    "reported_nfev": 64
  },
  {
    "group": "A1",
    "method": "Euler",
    "N": 128,
    "error": 0.001441725249405168,
    "order": 1.004728640335508,
    "nfev": 128,
    "reported_nfev": 128
  },
  {
    "group": "A1",
    "method": "RK4",
    "N": 16,
    "error": 4.9281128622347126e-08,
    "order": null,
    "nfev": 64,
    "reported_nfev": 64
  },
  {
    "group": "A1",
    "method": "RK4",
    "N": 32,
    "error": 3.000808712627645e-09,
    "order": 4.037612034611561,
    "nfev": 128,
    "reported_nfev"

## 3.2 热方程：解析解、离散模态与边界扩展
对零Dirichlet热方程，正弦模态连续解为 $e^{-\kappa\pi^2t}\sin(\pi x)$；FTCS离散模态因子为 $1-4r\sin^2(k\pi/(2N))$。一维均匀FTCS充分稳定条件为 $r\le1/2$，二维等网格等系数为 $r\le1/4$。这些条件不推广到所有方法。
非零Neumann测试属于A1公开函数分支核验：稳态 $u=x$ 满足 $u_t=u_{xx}=0$、左端导数1、右端值1。

In [4]:
A1_HEAT='def solve1d(N, r, T, u0=None, bc=(0.0, 0.0)):\n    # 一维显式：返回 (x, 终层解 U, M, dt, 实际 r)\n    h = 1.0 / N\n    M = max(1, int(round(T / (r * h * h))))\n    dt = T / M\n    r_eff = dt / (h * h)\n    x = np.linspace(0.0, 1.0, N + 1)\n    U = np.sin(np.pi * x) if u0 is None else np.asarray(u0, dtype=float).copy()\n    U[0], U[-1] = bc\n    for _ in range(M):\n        U[1:-1] = U[1:-1] + r_eff * (U[:-2] - 2.0 * U[1:-1] + U[2:])\n        U[0], U[-1] = bc\n    return x, U, M, dt, r_eff\n\n\ndef solve2d(N, r, T, x0=None):\n    # 二维显式：返回 (x, X, Y, 终层解 U, M, dt, 实际 r)\n    h = 1.0 / N\n    M = max(1, int(round(T / (r * h * h))))\n    dt = T / M\n    r_eff = dt / (h * h)\n    x = np.linspace(0.0, 1.0, N + 1)\n    X, Y = np.meshgrid(x, x, indexing="ij")\n    U = (np.sin(np.pi * X) * np.sin(np.pi * Y)) if x0 is None else x0(X, Y).astype(float)\n    for _ in range(M):\n        U[1:-1, 1:-1] = U[1:-1, 1:-1] + r_eff * (\n            U[:-2, 1:-1] + U[2:, 1:-1] + U[1:-1, :-2] + U[1:-1, 2:] - 4.0 * U[1:-1, 1:-1])\n    return x, X, Y, U, M, dt, r_eff'
A1_BOUNDARY='def assemble_1d(N, r, bc_left=("dirichlet", 0.0), bc_right=("dirichlet", 0.0), source=None):\n    # 构造隐式 Euler 的 (A, b)。bc: ("dirichlet", gamma) 或 ("neumann", sigma)\n    n = N - 1\n    A = np.zeros((n, n))\n    b = np.zeros(n)\n    for j in range(n):\n        A[j, j] = 1.0 + 2.0 * r\n        if j > 0:\n            A[j, j - 1] = -r\n        if j < n - 1:\n            A[j, j + 1] = -r\n    if source is not None:\n        b[:] = source\n    kind_l, val_l = bc_left\n    kind_r, val_r = bc_right\n    if kind_l == "dirichlet":\n        b[0] += r * val_l\n    else:                                   # Neumann：虚拟点 U_{-1} = U_1 - 2h*sigma\n        A[0, 0] = 1.0 + 2.0 * r\n        A[0, 1] = -2.0 * r\n    if kind_r == "dirichlet":\n        b[-1] += r * val_r\n    else:\n        A[-1, -1] = 1.0 + 2.0 * r\n        A[-1, -2] = -2.0 * r\n    return A, b\n\n\nN, r = 5, 0.4\nA, b = assemble_1d(N, r, ("dirichlet", 2.0), ("dirichlet", -1.0))\nnp.set_printoptions(precision=2, suppress=True)\nprint("A =")\nprint(A)\nprint("行和（内部行应为 1，首末行含边界项）：", A.sum(axis=1))\nprint("b 的首末分量（零旧值、无源项，只剩边界项）：", b[[0, -1]])'
A2_HEAT='# 3.2 FTCS 方法实现、运行与数值记录\ndef solve_ftcs(J, r, eps=0.0):\n    dx = L/J\n    dt = r*dx**2/kappa\n    steps = int(round(T/dt))\n    if not np.isclose(steps*dt, T, rtol=0, atol=1e-12):\n        raise ValueError("终止时间 T 必须为 Δt 的整数倍。")\n    x = np.linspace(0.0, L, J+1)\n    U = u_exact(x, 0.0, J, eps).copy()\n\n    phi = np.sin((J-1)*np.pi*x/L)\n    phi[0] = phi[-1] = 0.0\n    denominator = float(np.dot(phi[1:-1], phi[1:-1]))\n    ts, max_errs, rms_errs, max_temps, high_coeffs = [], [], [], [], []\n    for n in range(steps+1):\n        t = n*dt\n        ref = u_exact(x, t, J, eps)\n        delta = U[1:-1] - ref[1:-1]\n        ts.append(t)\n        max_errs.append(float(np.max(np.abs(U-ref))))\n        rms_errs.append(float(np.sqrt(np.mean(delta**2))))\n        max_temps.append(float(np.max(np.abs(U))))\n        high_coeffs.append(float(\n            np.dot(U[1:-1], phi[1:-1])/denominator\n        ))\n        if n < steps:\n            V = U.copy()\n            V[1:-1] = U[1:-1] + r*(U[2:] - 2*U[1:-1] + U[:-2])\n            V[0] = V[-1] = 0.0\n            U = V\n\n    return dict(\n        J=J, r=r, dx=dx, dt=dt, N=steps, x=x, U=U,\n        reference=u_exact(x, T, J, eps),\n        times=np.asarray(ts), E_inf_hist=np.asarray(max_errs),\n        E_rms_hist=np.asarray(rms_errs),\n        M_hist=np.asarray(max_temps),\n        high_coeff=np.asarray(high_coeffs),\n        E_inf=max_errs[-1], E_rms=rms_errs[-1]\n    )\n\nclock_start = time.perf_counter()\nstable = solve_ftcs(J=40, r=0.4, eps=epsilon)\nunstable = solve_ftcs(J=40, r=0.6, eps=epsilon)\nrefinements = [solve_ftcs(J, r=0.4) for J in (20, 40, 80, 160)]\nelapsed = time.perf_counter()-clock_start\n\nfor name, d in [("A/stable", stable), ("B/unstable", unstable)]:\n    print(f"{name}: r={d[\'r\']:.1f}, dx={d[\'dx\']:.5f} m, "\n          f"dt={d[\'dt\']:.8f} s, N={d[\'N\']}, "\n          f"E_inf={d[\'E_inf\']:.8e} deg C, "\n          f"E_rms={d[\'E_rms\']:.8e} deg C, "\n          f"M(T)={d[\'M_hist\'][-1]:.8e} deg C")\n\nprint("Refinement: J, dx(m), dt(s), N, E_inf(deg C), order")\nprevious_error = None\nfor d in refinements:\n    order = float("nan") if previous_error is None else np.log2(previous_error/d["E_inf"])\n    print(f"{d[\'J\']:3d}, {d[\'dx\']:.8f}, {d[\'dt\']:.9f}, {d[\'N\']:4d}, "\n          f"{d[\'E_inf\']:.8e}, {order:.5f}")\n    previous_error = d["E_inf"]\nprint(f"Numerical runtime without plotting: {elapsed:.4f} s")\n\n# 记录每个时间层的误差与高频模态系数。\nwith (data_dir/"A2组-3.2_timeseries.csv").open("w", encoding="utf-8", newline="") as f:\n    writer=csv.writer(f)\n    writer.writerow(["case","time_s","E_inf_degC","E_rms_degC",\n                     "max_abs_temp_degC","high_mode_coeff_degC"])\n    for name, d in [("stable_r0.4",stable),("unstable_r0.6",unstable)]:\n        for values in zip(d["times"],d["E_inf_hist"],d["E_rms_hist"],\n                          d["M_hist"],d["high_coeff"]):\n            writer.writerow([name,*values])\n\n# 记录不同网格上的误差及经验收敛阶。\nwith (data_dir/"A2组-3.2_refinement.csv").open("w", encoding="utf-8", newline="") as f:\n    writer=csv.writer(f)\n    writer.writerow(["J","dx_m","dt_s","steps","E_inf_degC","E_rms_degC","order"])\n    previous_error=None\n    for d in refinements:\n        order="" if previous_error is None else np.log2(previous_error/d["E_inf"])\n        writer.writerow([d["J"],d["dx"],d["dt"],d["N"],\n                         d["E_inf"],d["E_rms"],order])\n        previous_error=d["E_inf"]\n'

In [5]:
h1={'np':np};defs(A1_HEAT,h1);defs(A1_BOUNDARY,h1)
h2={'np':np,'L':1.,'kappa':.1,'A':1.,'T':.6};defs(A2_HEAT,h2)
def ref2(x,t,J,eps=0.):
    u=np.sin(np.pi*x)*np.exp(-.1*np.pi**2*t)+eps*np.sin((J-1)*np.pi*x)*np.exp(-.1*((J-1)*np.pi)**2*t)
    u[0]=u[-1]=0.;return u
h2['u_exact']=ref2
heat=[]
for group in ['A1','A2']:
 prev=None
 for N in [20,40,80]:
    if group=='A1':
        x,U,M,dt,r=h1['solve1d'](N,.4,.1);T=.1;kappa=1.
    else:
        d=h2['solve_ftcs'](N,.4);x,U,M,dt,r=d['x'],d['U'],d['N'],d['dt'],d['r'];T=.6;kappa=.1
    exact=np.exp(-kappa*np.pi**2*T)*np.sin(np.pi*x)
    err=float(np.max(abs(U-exact)));p=None if prev is None else math.log2(prev/err)
    discrete=(1-4*r*np.sin(np.pi/(2*N))**2)**M*np.sin(np.pi*x)
    defect=float(np.max(abs(U-discrete)))
    assert defect<2e-11
    if p is not None: assert 1.8<p<2.2
    heat.append(dict(group=group,N=N,T=T,kappa=kappa,dt=dt,r=r,error=err,order=p,discrete_defect=defect,boundary_error=float(max(abs(U[0]),abs(U[-1])))))
    prev=err
record('heat',heat)
unstable=h2['solve_ftcs'](40,.6,eps=1e-12)
record('CFL',dict(A2_unstable_max=float(np.max(abs(unstable['U']))),A2_r=.6,expected='高频扰动增长，属于预期失稳演示',one_dim_limit=.5,two_dim_limit=.25))
N=5;r=.4;x=np.linspace(0,1,N+1)
AA,bb=h1['assemble_1d'](N,r,('neumann',1.),('dirichlet',1.))
residual=AA@x[1:-1]-(x[1:-1]+bb)
record('A1_neumann',dict(N=N,r=r,reference='u=x，左端导数1，右端值1，稳态残差应为0',residual=residual.tolist(),max_residual=float(max(abs(residual))),tolerance=1e-12))
fig,axs=plt.subplots(1,2,figsize=(9,3.4))
for group in ['A1','A2']:
 rr=[a for a in heat if a['group']==group];axs[0].loglog([1/a['N'] for a in rr],[a['error'] for a in rr],'o-',label=group)
axs[0].legend();axs[0].set(xlabel='dx',ylabel='Max error')
axs[1].semilogy(unstable['times'],unstable['M_hist']);axs[1].set(xlabel='t',ylabel='max |U|, A2 r=0.6')
fig.tight_layout();fig.savefig(FIG/'B1组-热方程误差与失稳.png');plt.close(fig)

heat
[
  {
    "group": "A1",
    "N": 20,
    "T": 0.1,
    "kappa": 1.0,
    "dt": 0.001,
    "r": 0.3999999999999999,
    "error": 0.001062511783011033,
    "order": null,
    "discrete_defect": 1.3877787807814457e-15,
    "boundary_error": 0.0
  },
  {
    "group": "A1",
    "N": 40,
    "T": 0.1,
    "kappa": 1.0,
    "dt": 0.00025,
    "r": 0.3999999999999999,
    "error": 0.00026494995890052886,
    "order": 2.003687031794192,
    "discrete_defect": 1.4432899320127035e-15,
    "boundary_error": 0.0
  },
  {
    "group": "A1",
    "N": 80,
    "T": 0.1,
    "kappa": 1.0,
    "dt": 6.25e-05,
    "r": 0.3999999999999999,
    "error": 6.619528366480498e-05,
    "order": 2.0009195680862435,
    "discrete_defect": 1.0436096431476471e-14,
    "boundary_error": 0.0
  },
  {
    "group": "A2",
    "N": 20,
    "T": 0.6,
    "kappa": 0.1,
    "dt": 0.010000000000000002,
    "r": 0.4,
    "error": 0.0009466413487181358,
    "order": null,
    "discrete_defect": 1.1102230246251565e-15,
    

## 3.3 有限元：局部矩阵、解析误差与条件数
模型 $-u\prime\prime=\pi^2\sin(\pi x)$、零边界；弱形式 $\int u\prime v\prime=\int fv$。线性元局部刚度 $h^{-1}[[1,-1],[-1,1]]$。独立误差积分使用SciPy quad逐单元计算连续L2误差和H1半范数；容差针对两个积分算法的数值一致性，不要求离散误差为零。

In [6]:
A1_FEM='import warnings\nimport numpy as np\nimport matplotlib.pyplot as plt\nfrom pathlib import Path\n\nwarnings.filterwarnings("ignore", message=".*glyph.*")\n\nnp.random.seed(2026)\nplt.rcParams.update({\n    "font.family": "serif",\n    "font.serif": ["SimSun", "DejaVu Serif", "Times New Roman"],\n    "mathtext.fontset": "stix",\n    "axes.unicode_minus": False,\n    "figure.dpi": 120,\n})\n\nIMG = Path("图片"); DAT = Path("数据")\nIMG.mkdir(exist_ok=True); DAT.mkdir(exist_ok=True)\n\nGL = {\n    1: (np.array([0.0]), np.array([2.0])),\n    2: (np.array([-1/np.sqrt(3), 1/np.sqrt(3)]), np.array([1.0, 1.0])),\n    4: (np.array([-0.8611363115940526, -0.3399810435848563,\n                  0.3399810435848563, 0.8611363115940526]),\n        np.array([0.3478548451374538, 0.6521451548625461,\n                  0.6521451548625461, 0.3478548451374538])),\n}\n\n\ndef mesh(N):\n    return np.linspace(0.0, 1.0, N + 1)\n\n\ndef assemble(x, a=lambda t: 1.0, f=lambda t: 1.0, ngauss=4):\n    N = len(x) - 1\n    K = np.zeros((N + 1, N + 1))\n    F = np.zeros(N + 1)\n    xi, w = GL[ngauss]\n    for e in range(N):\n        h = x[e + 1] - x[e]\n        am = a(0.5 * (x[e] + x[e + 1]))\n        K[e:e + 2, e:e + 2] += am / h * np.array([[1.0, -1.0], [-1.0, 1.0]])\n        for q, wq in zip(xi, w):\n            xx = 0.5 * (x[e] + x[e + 1]) + 0.5 * h * q\n            phi = np.array([(1 - q) / 2, (1 + q) / 2])\n            F[e:e + 2] += 0.5 * h * wq * f(xx) * phi\n    return K, F\n\n\ndef solve(K, F):\n    u = np.zeros(len(F))\n    u[1:-1] = np.linalg.solve(K[1:-1, 1:-1], F[1:-1])\n    return u\n\n\ndef errors(x, u, ue, due):\n    N = len(x) - 1\n    xi, w = GL[4]\n    l2 = h1 = 0.0\n    for e in range(N):\n        h = x[e + 1] - x[e]\n        du = (u[e + 1] - u[e]) / h\n        for q, wq in zip(xi, w):\n            xx = 0.5 * (x[e] + x[e + 1]) + 0.5 * h * q\n            uh = u[e] * (1 - q) / 2 + u[e + 1] * (1 + q) / 2\n            l2 += 0.5 * h * wq * (ue(xx) - uh) ** 2\n            h1 += 0.5 * h * wq * (due(xx) - du) ** 2\n    return np.sqrt(l2), np.sqrt(h1)\n\n\ndef rate(e1, e2):\n    return np.log2(e1 / e2)\n\nprint("环境就绪：", np.__version__, plt.matplotlib.__version__)\n'
A2_FEM='# 第 1 个代码检查点：环境、模型、精确解\nimport sys\nimport numpy as np\nimport matplotlib\nimport matplotlib.pyplot as plt\nfrom numpy.polynomial.legendre import leggauss\n\ndef u_exact(x):\n    return np.sin(np.pi * x)\n\ndef du_exact(x):\n    return np.pi * np.cos(np.pi * x)\n\ndef source(x):\n    return np.pi**2 * np.sin(np.pi * x)\n\nprint("Python:", sys.version.split()[0])\nprint("NumPy:", np.__version__, "Matplotlib:", matplotlib.__version__)\nprint("u(0), u(0.5), u(1):", u_exact(np.array([0.0, 0.5, 1.0])))\nassert np.allclose(u_exact(np.array([0.0, 1.0])), [0.0, 0.0], atol=1e-14)\nprint("检查点 1 通过：模型及边界条件正确")\ndef element_matrices(x_left, x_right, rhs, n_gauss=4):\n    """连续分片线性单元的 2x2 刚度矩阵和 2 维载荷向量。"""\n    h_e = x_right - x_left\n    if h_e <= 0:\n        raise ValueError("单元长度必须为正")\n\n    K_e = np.array([[1.0, -1.0], [-1.0, 1.0]]) / h_e\n\n    xi, w = leggauss(n_gauss)\n    xq = (x_left + x_right) / 2 + (h_e / 2) * xi\n    N1 = (x_right - xq) / h_e\n    N2 = (xq - x_left) / h_e\n    fq = rhs(xq)\n\n    F_e = (h_e / 2) * np.array([\n        np.sum(w * fq * N1),\n        np.sum(w * fq * N2)\n    ])\n    return K_e, F_e\n\nK_example, F_example = element_matrices(\n    0.0, 0.5, lambda x: np.ones_like(x), n_gauss=4\n)\nprint("长度 1/2、f=1 的局部刚度矩阵：\\n", K_example)\nprint("局部载荷向量：", F_example)\nassert np.allclose(K_example, [[2, -2], [-2, 2]])\nassert np.allclose(F_example, [0.25, 0.25])\nprint("检查点 3 通过：与手算一致")\ndef assemble_fem(x, rhs, n_gauss=4):\n    """按单元装配全节点刚度矩阵、载荷向量（尚未施加边界条件）。"""\n    n_nodes = len(x)\n    K_all = np.zeros((n_nodes, n_nodes))\n    F_all = np.zeros(n_nodes)\n\n    for e in range(n_nodes - 1):\n        K_e, F_e = element_matrices(x[e], x[e + 1], rhs, n_gauss)\n        idx = [e, e + 1]  # 局部节点 -> 全局节点\n        K_all[np.ix_(idx, idx)] += K_e\n        F_all[idx] += F_e\n    return K_all, F_all\n\nx_two = np.array([0.0, 0.5, 1.0])\nf_one = lambda x: np.ones_like(x)\nK_two, F_two = assemble_fem(x_two, f_one)\nprint("两单元的完整装配矩阵：\\n", K_two)\nprint("两单元的完整载荷向量：", F_two)\n\nassert np.allclose(K_two, [[2, -2, 0], [-2, 4, -2], [0, -2, 2]])\nassert np.allclose(F_two, [0.25, 0.5, 0.25])\nprint("检查点 4 通过：共享节点的刚度和载荷正确累加")\ndef solve_fem(N, rhs=source, n_gauss=4, boundary=(0.0, 0.0)):\n    """装配 -> 消去 Dirichlet 边界自由度 -> 恢复全部节点值。"""\n    x = np.linspace(0.0, 1.0, N + 1)\n    K_all, F_all = assemble_fem(x, rhs, n_gauss)\n\n    U = np.zeros(N + 1)\n    U[0], U[-1] = boundary\n\n    K_II = K_all[1:-1, 1:-1]\n    right = (F_all[1:-1]\n             - K_all[1:-1, 0] * U[0]\n             - K_all[1:-1, -1] * U[-1])\n    U[1:-1] = np.linalg.solve(K_II, right)\n    residual = np.max(np.abs(K_II @ U[1:-1] - right))\n\n    return {\n        "x": x, "u": U, "K_all": K_all, "F_all": F_all,\n        "K_II": K_II, "rhs_I": right, "residual_inf": residual\n    }\n\ntwo = solve_fem(2, rhs=f_one)\nprint("两单元内部方程：", two["K_II"][0, 0], "* U1 =", two["rhs_I"][0])\nprint("中间节点 U1 =", two["u"][1], "；理论值 1/8 =", 1/8)\nprint("内部系统残差 =", two["residual_inf"])\n\nassert np.isclose(two["u"][1], 1/8)\nassert np.allclose(two["u"][[0, -1]], 0)\nassert np.allclose(two["K_all"], two["K_all"].T)\nassert np.linalg.eigvalsh(two["K_II"]).min() > 0\nassert two["residual_inf"] < 1e-12\nprint("检查点 5 通过：矩阵对称、内部正定、边界和残差正确")\ndef fem_integral_errors(x, U, n_gauss=12):\n    """在所有单元上积分，返回 (FEM_L2, FEM_H1_seminorm)。"""\n    xi, w = leggauss(n_gauss)\n    L2_sq = 0.0\n    H1_sq = 0.0\n    for e in range(len(x) - 1):\n        left, right = x[e], x[e + 1]\n        h_e = right - left\n        xq = (left + right) / 2 + (h_e / 2) * xi\n\n        t = (xq - left) / h_e\n        uh_q = (1 - t) * U[e] + t * U[e + 1]\n        duh_e = (U[e + 1] - U[e]) / h_e\n\n        L2_sq += (h_e / 2) * np.sum(w * (u_exact(xq) - uh_q)**2)\n        H1_sq += (h_e / 2) * np.sum(w * (du_exact(xq) - duh_e)**2)\n\n    return np.sqrt(L2_sq), np.sqrt(H1_sq)\n\nL2_8, H1_8 = fem_integral_errors(fem8["x"], fem8["u"])\nprint(f"N=8: FEM L2={L2_8:.9f}, FEM H1 seminorm={H1_8:.9f}")\nassert np.isclose(L2_8, 0.009920920, atol=1e-9)\nassert np.isclose(H1_8, 0.251181769, atol=1e-9)\nprint("检查点 8 通过：积分误差与教材第 19 页一致")'

In [7]:
from numpy.polynomial.legendre import leggauss
GL={n:leggauss(n) for n in [1,2,4]}
f1={'np':np,'GL':GL};defs(A1_FEM,f1)
f2={'np':np,'leggauss':leggauss};defs(A2_FEM,f2)
fem=[]
for group in ['A1','A2']:
 prev=None
 for N in [8,16,32,64]:
    if group=='A1':
        x=np.linspace(0,1,N+1);K,F=f1['assemble'](x,f=lambda t:np.pi**2*np.sin(np.pi*t));U=f1['solve'](K,F)
        native=f1['errors'](x,U,np.sin if False else lambda t:np.sin(np.pi*t),lambda t:np.pi*np.cos(np.pi*t))
    else:
        d=f2['solve_fem'](N);x,U,K,F=d['x'],d['u'],d['K_all'],d['F_all'];native=f2['fem_integral_errors'](x,U)
    l2=h1e=0.
    for e in range(N):
        a,b=x[e:e+2];s=(U[e+1]-U[e])/(b-a)
        l2+=quad(lambda z:(np.sin(np.pi*z)-(U[e]+s*(z-a)))**2,a,b,epsabs=1e-14)[0]
        h1e+=quad(lambda z:(np.pi*np.cos(np.pi*z)-s)**2,a,b,epsabs=1e-14)[0]
    l2,h1e=np.sqrt(l2),np.sqrt(h1e)
    assert max(abs(l2-native[0]),abs(h1e-native[1]))<1e-9
    pp=None if prev is None else [math.log2(prev[0]/l2),math.log2(prev[1]/h1e)]
    if pp: assert 1.8<pp[0]<2.2 and .9<pp[1]<1.1
    fem.append(dict(group=group,N=N,dof=N-1,L2=l2,H1_seminorm=h1e,orders=pp,residual=float(max(abs(K[1:-1,1:-1]@U[1:-1]-F[1:-1])))))
    prev=l2,h1e
record('FEM',fem)
Kloc,Floc=f2['element_matrices'](0.,.5,lambda x:np.ones_like(x))
assert np.allclose(Kloc,[[2,-2],[-2,2]],atol=1e-12) and np.allclose(Floc,[.25,.25],atol=1e-12)
linear=f2['solve_fem'](8,rhs=lambda x:np.zeros_like(x),boundary=(1.,3.))
record('FEM_boundary',dict(local_K=Kloc.tolist(),local_F=Floc.tolist(),A2_linear_error=float(max(abs(linear['u']-(1+2*linear['x']))))))
K,F=f1['assemble'](np.linspace(0,1,9),f=lambda x:np.pi**2*np.sin(np.pi*x))
Kb=K.copy();Kb[0,0]=Kb[-1,-1]=1e12
record('A1_condition',dict(original_internal=float(np.linalg.cond(K[1:-1,1:-1])),penalized_internal=float(np.linalg.cond(Kb[1:-1,1:-1])),penalized_full=float(np.linalg.cond(Kb)),reason='A1单元61打印内部子块，未包含被置大数的边界行列'))
for group in ['A1','A2']:
 rr=[a for a in fem if a['group']==group]
 plt.loglog([1/a['N'] for a in rr],[a['L2'] for a in rr],'o-',label=group+' L2')
 plt.loglog([1/a['N'] for a in rr],[a['H1_seminorm'] for a in rr],'s--',label=group+' H1 seminorm')
plt.legend();plt.xlabel('h');plt.ylabel('Integral error');plt.tight_layout();plt.savefig(FIG/'B1组-有限元误差.png');plt.close()

FEM
[
  {
    "group": "A1",
    "N": 8,
    "dof": 7,
    "L2": 0.009920919910041605,
    "H1_seminorm": 0.2511817693763456,
    "orders": null,
    "residual": 8.881784197001252e-16
  },
  {
    "group": "A1",
    "N": 16,
    "dof": 15,
    "L2": 0.0024865013394177998,
    "H1_seminorm": 0.12583315847452248,
    "orders": [
      1.9963566913712616,
      0.9972196191430991
    ],
    "residual": 3.9968028886505635e-15
  },
  {
    "group": "A1",
    "N": 32,
    "dof": 31,
    "L2": 0.0006220177931452975,
    "H1_seminorm": 0.0629469052002624,
    "orders": [
      1.999089453178313,
      0.9993047845850747
    ],
    "residual": 8.43769498715119e-15
  },
  {
    "group": "A1",
    "N": 64,
    "dof": 63,
    "L2": 0.00015552898472378875,
    "H1_seminorm": 0.03147724465069757,
    "orders": [
      1.9997723807606032,
      0.9998261886589844
    ],
    "residual": 2.0594637106796654e-14
  },
  {
    "group": "A2",
    "N": 8,
    "dof": 7,
    "L2": 0.009920919910041605,
    "H1

## 3.4 已知方程族的参数拟合与独立评估
A1选取logistic参数辨识，训练21点[0,2]、种子2024、相对噪声1%；评估301点[0,3]及初值0.02/0.05/0.2。A2选取 $y\prime=-ay$ 单参数训练，训练11点[0,2]、初值0.8/1/1.2，评估101点及初值0.9/1.4/2。均已给定方程族，不能冒称完整多层网络或完整PDE PINN训练。

In [8]:
A1_LOGISTIC='# ---------------------------------------------------------------- 数据与模型\nR_TRUE, K_TRUE, U0_TRUE = 0.8, 1.0, 0.05\n\n\ndef logistic(t, r=R_TRUE, K=K_TRUE, u0=U0_TRUE):\n    \'\'\'logistic 真解（也用来生成数据）。\'\'\'\n    return K / (1.0 + (K / u0 - 1.0) * np.exp(-r * t))\n\n\ndef make_data(seed=2024, noise=0.01, n=21):\n    \'\'\'21 个等距观测点，加 1% 相对噪声（固定随机种子，结果可复现）。\'\'\'\n    rng = np.random.default_rng(seed)\n    t = np.linspace(0, 2, n)\n    u = logistic(t) * (1 + rng.uniform(-noise, noise, n))\n    return t, u\n\n\ndef integrate_logistic(r, K, u0, t, h=0.1, method="rk4"):\n    \'\'\'把 u\'=r u(1-u/K) 从 u0 积分到各个时刻 t_i（步长 h）。\'\'\'\n    out = np.empty_like(t)\n    u, cur, k = u0, 0.0, 0\n    f = lambda x: r * x * (1.0 - x / K)\n    while k < len(t):\n        while cur < t[k] - 1e-12:\n            step = min(h, t[k] - cur)\n            if method == "euler":\n                u = u + step * f(u)\n            else:\n                k1 = f(u)\n                k2 = f(u + step * k1 / 2)\n                k3 = f(u + step * k2 / 2)\n                k4 = f(u + step * k3)\n                u = u + step * (k1 + 2 * k2 + 2 * k3 + k4) / 6\n            cur += step\n        out[k] = u\n        k += 1\n    return out\n\n\ndef gauss_newton(t, u_obs, h=0.1, method="rk4", p0=(1.0, 1.10, 0.05), iters=60):\n    \'\'\'对 (r, K, u0) 做 Gauss–Newton（数值雅可比 + Levenberg 阻尼）。\'\'\'\n    p = np.array(p0, float)\n    lam = 1e-9\n    resid = lambda q: integrate_logistic(*q, t, h, method) - u_obs\n    r0 = resid(p)\n    cost = r0 @ r0\n    for _ in range(iters):\n        J = np.empty((len(t), 3))\n        for j in range(3):\n            dp = np.zeros(3)\n            dp[j] = 1e-6 * max(1.0, abs(p[j]))\n            J[:, j] = (resid(p + dp) - r0) / dp[j]\n        step = -np.linalg.solve(J.T @ J + lam * np.eye(3), J.T @ r0)\n        r_new = resid(p + step)\n        c_new = r_new @ r_new\n        if c_new < cost:\n            p, r0, cost = p + step, r_new, c_new\n            lam = max(lam * 0.3, 1e-12)\n        else:\n            lam *= 10\n            if lam > 1e8:\n                break\n        if np.linalg.norm(step) < 1e-11:\n            break\n    return p, cost\n\n\nt_obs, u_obs = make_data()\nprint("前 5 个观测点：", np.round(u_obs[:5], 5), "…")\n\nfor h_, m_ in [(0.2, "euler"), (0.1, "rk4"), (0.05, "rk4")]:\n    p_, loss_ = gauss_newton(t_obs, u_obs, h=h_, method=m_)\n    print(f"{m_.upper():<5} h={h_:<5} → r={p_[0]:.4f}  K={p_[1]:.4f}  u0={p_[2]:.5f}  "\n          f"loss={loss_:.2e}")\n\nR_HAT, K_HAT, U0_HAT = gauss_newton(t_obs, u_obs, 0.1, "rk4")[0]\nprint(f"\\n取 RK4(h=0.1) 的结果：r̂={R_HAT:.3f}  K̂={K_HAT:.3f}  û0={U0_HAT:.4f}")'
A2_LEARN='import numpy as np\nfrom scipy.integrate import solve_ivp\nfrom scipy.optimize import minimize_scalar\n\nA_TRUE, T, H = 1.6, 2.0, 0.2\nTRAIN_Y0 = np.array([0.8, 1.0, 1.2])\nTEST_Y0 = np.array([0.9, 1.4, 2.0])\nT_TRAIN = np.linspace(0, T, 11)\nT_TEST = np.linspace(0, T, 101)\n\ndef amplification(a, h, solver):\n    z = -a * h\n    if solver == "Euler":\n        return 1 + z\n    return 1 + z + z**2/2 + z**3/6 + z**4/24\n\ndef train_mse(a, solver):\n    factors = amplification(a, H, solver) ** np.arange(len(T_TRAIN))\n    pred = TRAIN_Y0[:, None] * factors[None, :]\n    exact = TRAIN_Y0[:, None] * np.exp(-A_TRUE * T_TRAIN)[None, :]\n    return np.mean((pred - exact)**2)\n\nfor solver in ["Euler", "RK4"]:\n    result = minimize_scalar(\n        lambda a: train_mse(a, solver),\n        bounds=(0.5, 2.5), method="bounded",\n        options={"xatol": 1e-14},\n    )\n    a_fit = result.x\n    pred = np.stack([\n        solve_ivp(\n            lambda t, y: -a_fit*y, (0, T), [y0],\n            t_eval=T_TEST, method="DOP853",\n            rtol=1e-12, atol=1e-14,\n        ).y[0]\n        for y0 in TEST_Y0\n    ])\n    exact = TEST_Y0[:, None] * np.exp(-A_TRUE * T_TEST)[None, :]\n    diff = pred - exact\n    print(solver, "a=", a_fit,\n          "train MSE=", train_mse(a_fit, solver),\n          "test RMSE=", np.sqrt(np.mean(diff**2)),\n          "test max error=", np.max(np.abs(diff)))\n    print("Final-point errors (y0=1.4):")\n    for h in [0.2, 0.1, 0.05]:\n        n = round(T / h)\n        yn = 1.4 * amplification(a_fit, h, solver)**n\n        print(" h=", h, " error=", abs(yn-1.4*np.exp(-A_TRUE*T)))\n\nprint("Poisson finite-difference errors:")\nfor N in [5, 10, 20, 40]:\n    dx = 1.0/N\n    x = np.arange(N+1)*dx\n    coefficient = np.pi**2*dx**2/(4*np.sin(np.pi*dx/2)**2)\n    approximate = coefficient*np.sin(np.pi*x)\n    error = np.max(np.abs(approximate-np.sin(np.pi*x)))\n    print("N=", N, "dx=", dx, "max error=", error)'

In [9]:
l1={'np':np,'R_TRUE':.8,'K_TRUE':1.,'U0_TRUE':.05};defs(A1_LOGISTIC,l1)
tobs,uobs=l1['make_data']();pars,cost=l1['gauss_newton'](tobs,uobs,h=.1,method='rk4')
grid=np.linspace(0,3,301);logistic=[]
for u0 in [.02,.05,.2]:
    numeric=solve_ivp(lambda t,u:pars[0]*u*(1-u/pars[1]),(0,3),[u0],t_eval=grid,method='DOP853',rtol=1e-11,atol=1e-13).y[0]
    closed=l1['logistic'](grid,*pars[:2],u0)
    truth=l1['logistic'](grid,u0=u0)
    defect=float(max(abs(numeric-closed)));assert defect<1e-9
    logistic.append(dict(y0=u0,max_error=float(max(abs(numeric-truth))),solver_vs_closed=defect))
record('A1_logistic',dict(parameters=pars.tolist(),train_SSE=float(cost),seed=2024,train_grid=[0,2,21],eval_grid=[0,3,301],tests=logistic))
l2={'np':np,'H':.2,'A_TRUE':1.6,'T_TRAIN':np.linspace(0,2,11),'TRAIN_Y0':np.array([.8,1.,1.2])};defs(A2_LEARN,l2)
learn=[];t=np.linspace(0,2,101);ys=np.array([.9,1.4,2.])
for method in ['Euler','RK4']:
 opt=minimize_scalar(lambda a:l2['train_mse'](a,method),bounds=(.5,2.5),method='bounded',options={'xatol':1e-14})
 expected=(1-math.exp(-1.6*.2))/.2 if method=='Euler' else None
 if expected is not None: assert abs(opt.x-expected)<1e-7
 pred=ys[:,None]*np.exp(-opt.x*t);truth=ys[:,None]*np.exp(-1.6*t)
 learn.append(dict(method=method,a_fit=float(opt.x),analytic_fit=expected,train_MSE=float(opt.fun),test_RMSE=float(np.sqrt(np.mean((pred-truth)**2))),test_max_error=float(np.max(abs(pred-truth)))))
record('A2_learn',learn)
record('PINN_counterexample',dict(equation='u_prime=u, u(0)=1',candidate='u=1+t',residual_at_training_t0=0.,error_at_t1=math.e-2,conclusion='单配点残差为零不推出轨迹正确；这是解析反例，不是网络训练结果'))
plt.bar(['Euler','RK4'],[q['test_RMSE'] for q in learn]);plt.yscale('log');plt.ylabel('Continuous-model test RMSE');plt.tight_layout();plt.savefig(FIG/'B1组-参数拟合评估.png');plt.close()
Path('B1组-测试结果.json').write_text(json.dumps(RESULTS,ensure_ascii=False,indent=2),encoding='utf-8')
print('所有正常基准断言完成；已确认差异见Neumann与条件数记录。学生签认：待完成。')

A1_logistic
{
  "parameters": [
    0.8104019590612151,
    0.9221075324249774,
    0.04974290121777655
  ],
  "train_SSE": 9.969050462167035e-06,
  "seed": 2024,
  "train_grid": [
    0,
    2,
    21
  ],
  "eval_grid": [
    0,
    3,
    301
  ],
  "tests": [
    {
      "y0": 0.02,
      "max_error": 0.002029922948995455,
      "solver_vs_closed": 8.269968043705944e-12
    },
    {
      "y0": 0.05,
      "max_error": 0.0032185894999231524,
      "solver_vs_closed": 5.448530515650418e-12
    },
    {
      "y0": 0.2,
      "max_error": 0.03384076089211774,
      "solver_vs_closed": 5.523137502905229e-12
    }
  ]
}
A2_learn
[
  {
    "method": "Euler",
    "a_fit": 1.3692548116030006,
    "analytic_fit": 1.3692548146315457,
    "train_MSE": 4.771674275900165e-19,
    "test_RMSE": 0.06538949565804164,
    "test_max_error": 0.11446598404650477
  },
  {
    "method": "RK4",
    "a_fit": 1.6001828356812346,
    "analytic_fit": null,
    "train_MSE": 2.682713467712476e-18,
    "test_RM